Daily Challenge: Data Handling and Analysis in Python


What You Will Learn

    Advanced techniques for data normalization, reduction, and aggregation.
    Skills in gathering, exploring, integrating, and cleaning data using Python.
    Proficiency in using Pandas for complex data manipulation.


Your Task

    Download and import the Data Science Job Salary dataset.
    Normalize the ‘salary’ column using Min-Max normalization which scales all salary values between 0 and 1.
    Implement dimensionality reduction like Principal Component Analysis (PCA) or t-SNE to reduce the number of features (columns) in the dataset.
    Group the dataset by the ‘experience_level’ column and calculate the average and median salary for each experience level (e.g., Junior, Mid-level, Senior).

Hint :

    As a reminder, normalization is crucial when dealing with data that has different ranges. For example, salary data might have a wide range (e.g., from $20,000 to $200,000). By scaling the data using Min-Max normalization, you make sure that all salary values fall within a consistent range (0 to 1). This is particularly helpful when the data is going to be used in machine learning models, as some algorithms (like k-nearest neighbors or neural networks) perform better when features are normalized. It ensures that no single salary dominates the learning process, making the analysis more balanced.

    Dimensionality reduction helps simplify complex datasets by reducing the number of variables under consideration. This can make the data more manageable and help avoid the curse of dimensionality—a phenomenon where machine learning models struggle when dealing with high-dimensional data.
    PCA, for instance, helps in retaining the most important information (variance) from the dataset while reducing noise and redundancy.
    It can also speed up the training process for models and help in visualizing data in fewer dimensions.

    Aggregating data helps in understanding trends within subgroups of the dataset.
    Calculating average and median salaries for each experience level gives insights into the compensation distribution and disparities across different job levels. This kind of aggregation can help in answering business questions like “How does salary evolve with experience?” or “What is the salary distribution for senior-level roles?”




In [1]:
import pandas as pd
    # Download and import the Data Science Job Salary dataset.
##df1 will be raw data
df1 = pd.read_csv('datascience_salaries.csv')
#view dataset before any action:
df1.head(10)


,Unnamed: 0,job_title,job_type,experience_level,location,salary_currency,salary
0,0,Data scientist,Full Time,Senior,New York City,USD,149000
1,2,Data scientist,Full Time,Senior,Boston,USD,120000
2,3,Data scientist,Full Time,Senior,London,USD,68000
3,4,Data scientist,Full Time,Senior,Boston,USD,120000
4,5,Data scientist,Full Time,Senior,New York City,USD,149000
5,6,Data scientist,Full Time,Senior,London,USD,68000
6,7,Data scientist,Full Time,Senior,Research Triangle Park,USD,69000
7,8,Data scientist,Full Time,Senior,Sydney,USD,68000
8,9,Data scientist,Full Time,Senior,San Francisco,USD,140000
9,10,Data scientist,Full Time,Senior,Sofia,USD,68000


In [2]:
    # Normalize the ‘salary’ column using Min-Max normalization which scales all salary values between 0 and 1.
#First copying data to df2 as a new dataframe for analysis
df2 = df1.copy()
#Now using MinMaxScaler to normalize 'salary' column:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df2['salary_normalized'] = scaler.fit_transform(df2[['salary']])

print(df2[['salary', 'salary_normalized']].head())

   salary  salary_normalized
0  149000           0.601010
1  120000           0.454545
2   68000           0.191919
3  120000           0.454545
4  149000           0.601010


In [3]:
    # Implement dimensionality reduction like Principal Component Analysis (PCA) or t-SNE to reduce the number of features (columns) in the dataset.
#This can only work when all columns are numerical. First let's examine the columns to see which are relevant for analysis and can be encoded to numerical columns.
df2['job_title'].value_counts()

job_title
Data scientist      394
Data analyst        368
Machine learning    289
Big data            101
ML Ops               19
Name: count, dtype: int64

In [4]:
df2['job_type'].value_counts()

job_type
Full Time     1136
Internship      35
Name: count, dtype: int64

In [5]:
df2['experience_level'].value_counts()

experience_level
Senior       727
Mid          305
Entry        126
Executive     13
Name: count, dtype: int64

In [6]:
df2['location'].value_counts()


location
London                                   75
Remote                                   50
San Francisco                            43
Bengaluru                                34
Paris                                    33
                                         ..
Ballerup                                  1
Ho Chi Minh City                          1
UK/EU or compatible timezone (Remote)     1
North Sydney                              1
New York City / Remote                    1
Name: count, Length: 320, dtype: int64

In [7]:
df2['salary_currency'].value_counts()

salary_currency
USD    1157
EUR       9
GBP       5
Name: count, dtype: int64

In [8]:
#To set up PCA all columns must be numeric. Although one-hot encoding is usually used for categorical values, to minimize column amount we will use label encoding for this purpose as we have many categorical values. So doing label encoding for job_title, job_type, experience_level, location, salary_currency.
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df2['job_title'] = le.fit_transform(df2['job_title'])
df2['job_type'] = le.fit_transform(df2['job_type'])
df2['experience_level'] = le.fit_transform(df2['experience_level'])
df2['location'] = le.fit_transform(df2['location'])
df2['salary_currency'] = le.fit_transform(df2['salary_currency'])



In [9]:
#Code for PCA:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df2_scaled = scaler.fit_transform(df2)

pca = PCA(n_components=0.95)

df2_pca = pca.fit_transform(df2_scaled)

print(f"Original shape: {df2.shape}")   
print(f"Reduced shape: {df2_pca.shape}")

Original shape: (1171, 8)
Reduced shape: (1171, 7)


In [11]:
#Group the dataset by the ‘experience_level’ column and calculate the average and median salary for each experience level (e.g., Junior, Mid-level, Senior).
#Starting with the df1 original dataframe which has the regular non-normalized salary. But first limiting the data to rows with USD in salary, as there are a very small amount of rows with salary in other units that should not be aggregated with USD salary figures.
df3 = df1.copy()
df3 = df3[df3['salary_currency'] == 'USD']

# Group by experience level and calculate mean and median
salary_stats = df3.groupby('experience_level')['salary'].agg(['mean', 'median'])

# Rename columns for clarity
salary_stats.columns = ['Average Salary', 'Median Salary']

print(salary_stats)

                  Average Salary  Median Salary
experience_level                               
Entry               36111.111111        30000.0
Executive           76076.923077        46000.0
Mid                 51812.709030        51000.0
Senior              75403.337969        68000.0


Commentary: It is surprising that the median salary for Executives is lower than that of Mid and Senior employees, while the average salary for Executives remains higher than both. It's not clear why that would be. It's possible that there are many small firms with only Entry and Executive employees, where Executives are at a pay level one above Entry. Further analysis could look into this.